In [1]:
from time import sleep as delay
from project.el5600 import project
if "ic" in globals():
    ic.close()
ic = project(device="el5600", revision="a1", emulator="cp2112", logging=False, is_gui=False)

from phy.multimeter import keithley_dm6500, rohde_hmc8012, dmm
from phy.power_supply import keithley_2470, keysight_e36232a
from phy.scope import tektronix_dp_series_usb
from phy.eloader import sdl1030x
# from phy.relay_16ch import relay_box
# from phy.chamber_th3 import chamber

%matplotlib tk
from interface.docs.output_chart import plot
from interface.cui_colors import color
from interface.i2c_bridge.tca9548a import tca9548
from interface.docs.output_excel import excel_frame, style
from interface.cui_logger import logger as log

import numpy as np

chart = plot()

dm1 = keithley_dm6500()
dm2 = rohde_hmc8012()
ps = keysight_e36232a()
sm = keithley_2470()
ld = sdl1030x()
ds = tektronix_dp_series_usb()

# relay = relay_box(i2c_h=ic)
# tc = chamber(port=3)
# mux = tca9548(ic.i2c_h, 0x70)

# --------------------------------------------------
# list_voltage = list(np.arange(5, 8, 0.005))
# voltage  = [round(num, 3) for num in list_voltage]
# --------------------------------------------------

initialized the dm6500 connection
initialized the hmc 8012 connection
initialized the keysight e36232a connection
initialized the 2470 source meter
initialized the e-loader connection
initialized the scope connection


In [ ]:
def disable_all():
    ld.disable
    sm.disable
    ps.disable
    
    ld.iset = 1
    sm.vset = 3.3
    ps.vset = 5.0

In [4]:
disable_all()

h = ic.get_i2c_handler()
h.gpio_7(0) # EN pin control

ld.disable
ps.disable
sm.cfg_all = 3.3, 0.1
sm.power_recycle

initial_isgn = ic.trm_isgn
initial_isofs = ic.trm_isofs

print(f"initial_isgn  : {initial_isgn}")
print(f"initial_isofs : {initial_isofs}")

# disconnect EN
# activate tst mode
ic.write_byte(0xa0, 0xf9)
ic.write_byte(0xa0, 0x9f)
print(f"Read byte : [0xa0]={ic.read_byte(0xa0):#04x}")

initial_isgn  : 28
initial_isofs : 4
Read byte : [0xa0]=0x01


In [5]:
# pre-load
ic.write_byte(0xc0, 0x37)
ic.write_byte(0xc4, 0x01)
ic.write_byte(0xc5, 0x38)

ic.write_byte(0xa1, 0x31)
ic.write_byte(0xa1, 0x00)

ic.write_byte(0xa2, 0x31)
ic.write_byte(0xa2, 0x00)

ic.write_byte(0xa1, 0x62)
ic.write_byte(0xa7, 0x80)
ic.write_byte(0xa7, 0xc0)
ic.write_byte(0xa3, 0x10)
ic.write_byte(0xa1, 0x21)
ic.write_byte(0xa1, 0x22)
ic.write_byte(0xa3, 0x18)

ic.write_byte(0xbd, 0x04)
ic.write_byte(0xa3, 0x00)
ic.write_byte(0xa3, 0x04)
ic.write_byte(0xa3, 0x00)

ic.write_byte(0xa1, 0x32)
ic.write_byte(0xa1, 0x21)
ic.write_byte(0Xa1, 0x52)

ic.write_byte(0xa1, 0x00)
ic.write_byte(0xa7, 0x00)

ps.vset = 5
ps.power_recycle

ic.write_byte(0xbd, 0x00)
ic.write_byte(0xa2, 0x44)
ic.write_byte(0x0c, 0x40)

ic.write_byte(0xa2, 0x04)

In [6]:
h.gpio_7(1)

In [7]:
# trm_isgn[5:0]
# trm_isofs[2:0]

ps.cfg_all = 20, 5

ic.write_byte(0xa6, 0x10)
ic.write_byte(0xc5, 0xa8)
ic.write_byte(0xa2, 0x24)
ic.write_byte(0xca, 0x9f)
ic.trm_isofs = 4

In [ ]:
# --------------------------------------------------
evb_no = "evb#1"
# --------------------------------------------------

from datetime import datetime
date_str = datetime.now().strftime("%Y%m%d")

# change the current scale to 10A
dm2.current_10A

ld.iset = 0.01
ld.power_recycle

log.output_set_filename(f"[{date_str}_EL6200_A1] trm_isgn - {evb_no}")

# change the current scale to 10A
dm2.current_10A

ld.iset = 0.5
ld.power_recycle

log.output_csv(["trm_isgn", "trm_isofs"])
log.output_csv([initial_isgn, initial_isofs])


print(f"[initial value] trm_isgn  : {initial_isgn}")
print(f"[initial value] trm_isofs : {initial_isofs}")

ic.trm_isgn = 0
ic.trm_isofs = 0

print(f"reset trm_isgn  : {initial_isgn:#x} --> {ic.trm_isgn:#x}")
print(f"reset trm_isofs : {initial_isofs:#x} --> {ic.trm_isofs:#x}")

In [ ]:
for trim_bit in range(1):
    
    ic.trm_isgn = trim_bit
    ic.write_byte(0x0c, 0x40)

    list_item = list(np.arange(1.0, 5, 0.05))
    for_item  = [round(num, 2) for num in list_item]

    coarse = 0
    print("find start trim bit for time save")

    for item_i in for_item:
        
        ld.iset = item_i
        
        read_i = abs(round(dm2.current_10A, 6))
        read_cocp = abs(round(dm1.voltage_10, 1))
        print(f"[{trim_bit:#04x}] {read_i}, {read_cocp}")
        
        ret = [read_i, read_cocp]
        
        if read_cocp > 0.5:
            ld.iset = 0.5
            break
        else:
            coarse = round(read_i - 0.1, 6)

    ld.iset = 0.5
    print(f"[{trim_bit:#04x}] coarse current : {coarse}")

    # --------------------------------------------------

    list_item = list(np.arange(coarse, 5, 0.005))
    for_item  = [round(num, 3) for num in list_item]
    
    pre_isns_lo = 0
    print("find pre_isns_lo value")
    delay(1)

    for item_i in for_item:

        ld.iset = item_i
        
        read_i = abs(round(dm2.current_10A, 6))
        read_cocp = abs(round(dm1.voltage_10, 1))
        print(f"[{trim_bit:#04x}] {read_i}, {read_cocp}")
        
        ret = [read_i, read_cocp]
        
        if read_cocp > 0.5:
            ld.iset = 0.5
            break
        else:        
            pre_isns_lo = read_i
        
    ld.iset = 0.5
    print(f"[{trim_bit:#04x}] isns_lo : {pre_isns_lo}")
    
    # --------------------------------------------------

    previous_0Ch = ic.read_byte(0x0c)
    print(f"0x0c : {previous_0Ch:#04x}")
    ic.write_byte(0x0c, 0x5f)
    print(f"0x0c : {ic.read_byte(0x0c):#04x}")

    list_item = list(np.arange(2.2, 5, 0.05))
    for_item  = [round(num, 2) for num in list_item]

    coarse = 0
    print("find coarse value")
    delay(1)

    for item_i in for_item:

        ld.iset = item_i
        
        read_i = abs(round(dm2.current_10A, 6))
        read_cocp = abs(round(dm1.voltage_10, 1))
        print(f"[{trim_bit:#04x}] {read_i}, {read_cocp}")
        
        ret = [read_i, read_cocp]
        
        if read_cocp > 0.5:
            ld.iset = 0.5
            break
        else:
            coarse = round(read_i - 0.1, 6)

    print(f"coarse : {coarse}")
    ld.iset = 0.5
    
    # --------------------------------------------------

    list_item = list(np.arange(coarse, 5, 0.005))
    for_item  = [round(num, 3) for num in list_item]

    pre_isns_hi = 0
    print("find pre_isns_hi value")
    delay(1)    
    
    for item_i in for_item:

        ld.iset = item_i
        
        read_i = round(dm2.current_10A, 6)
        read_cocp = abs(round(dm1.voltage_10, 1))
        print(f"[{trim_bit:#04x}] {read_i}, {read_cocp}")
        
        ret = [read_i, read_cocp]
        
        if read_cocp > 0.5:
            ld.iset = 0.5
            break
        else:
            pre_isns_hi = read_i

    ld.iset = 0.5
    print(f"isns_hi : {pre_isns_hi}")

    isns_delta = round((pre_isns_hi - pre_isns_lo), 6)

    print(f"[isgn={trim_bit:#04x}, isofs={ic.trm_isofs}] isns_delta (target {round(1.003*1.85, 5)}) : {isns_delta:.05f}")
    
    if isns_delta > 1.9:
        break

In [ ]:
# adjust the start bit for trim
# check the inital isns_delta
# 0.1 isns_delta / 10 trim code
# e.g. : if code 0x00 and delta is 1.5, code 25d will be delta 1.75
start_fine_trim = 25

In [ ]:
log.output_csv([])
log.output_csv([])
log.output_csv([])
log.output_csv(["trm_isgn", "trm_isofs", "isns_lo", "isns_hi", "isns_delta"])

for trim_bit in range(start_fine_trim, 0b1_000_000):
    
    ic.trm_isgn = trim_bit
    ic.write_byte(0x0c, 0x40)

    list_item = list(np.arange(1.0, 5, 0.05))
    for_item  = [round(num, 2) for num in list_item]

    coarse = 0
    print("find coarse value")

    for item_i in for_item:
        # ld.vset = item_i
        ld.iset = item_i
        
        read_i = abs(round(dm2.current_10A, 6))
        read_cocp = abs(round(dm1.voltage_10, 1))
        print(f"[{trim_bit:#04x}] {read_i}, {read_cocp}")
        
        ret = [read_i, read_cocp]
        
        if read_cocp > 0.5:
            ld.iset = 0.5
            break
        else:
            coarse = round(read_i - 0.1, 6)

    ld.iset = 0.5
    print(f"[{trim_bit:#04x}] coarse current : {coarse}")

    # --------------------------------------------------

    list_item = list(np.arange(coarse, 5, 0.003))
    for_item  = [round(num, 3) for num in list_item]
    
    pre_isns_lo = 0
    print("find pre_isns_lo value")
    delay(1)

    for item_i in for_item:
        ld.iset = item_i
        
        read_i = abs(round(dm2.current_10A, 6))
        read_cocp = abs(round(dm1.voltage_10, 1))
        print(f"[{trim_bit:#04x}] {read_i}, {read_cocp}")
        
        ret = [read_i, read_cocp]
        
        if read_cocp > 0.5:
            ld.iset = 0.5
            break
        else:        
            pre_isns_lo = read_i
        
    ld.iset = 0.5
    print(f"[{trim_bit:#04x}] isns_lo : {pre_isns_lo}")
    
    # --------------------------------------------------

    previous_0Ch = ic.read_byte(0x0c)
    print(f"0x0c : {previous_0Ch:#04x}")
    ic.write_byte(0x0c, 0x5f)
    print(f"0x0c : {ic.read_byte(0x0c):#04x}")

    list_item = list(np.arange(2.2, 5, 0.05))
    for_item  = [round(num, 2) for num in list_item]

    coarse = 0
    print("find coarse value")
    delay(1)

    for item_i in for_item:

        ld.iset = item_i
        
        read_i = abs(round(dm2.current_10A, 6))
        read_cocp = abs(round(dm1.voltage_10, 1))
        print(f"[{trim_bit:#04x}] {read_i}, {read_cocp}")
        
        ret = [read_i, read_cocp]
        
        if read_cocp > 0.5:
            ld.iset = 0.5
            break
        else:
            coarse = round(read_i - 0.1, 6)

    print(f"coarse : {coarse}")
    ld.iset = 0.5
    
    # --------------------------------------------------

    list_item = list(np.arange(coarse, 5, 0.003))
    for_item  = [round(num, 3) for num in list_item]

    pre_isns_hi = 0
    print("find pre_isns_hi value")
    delay(1)    
    
    for item_i in for_item:

        ld.iset = item_i
        
        read_i = round(dm2.current_10A, 6)
        read_cocp = abs(round(dm1.voltage_10, 1))
        print(f"[{trim_bit:#04x}] {read_i}, {read_cocp}")
        
        ret = [read_i, read_cocp]
        
        if read_cocp > 0.5:
            ld.iset = 0.5
            break
        else:
            pre_isns_hi = read_i

    ld.iset = 0.5
    print(f"isns_hi : {pre_isns_hi}")

    isns_delta = round((pre_isns_hi - pre_isns_lo), 6)

    print(f"[{trim_bit:#04x}] isns_delta (target {round(1.003*1.85, 5)}) : {isns_delta:.05f}")
    
    isofs = ic.trm_isofs
    log.output_csv([trim_bit, isofs, pre_isns_lo, pre_isns_hi, isns_delta])
    
    if isns_delta > 1.87:
        break

In [ ]:
# overwrite the trm_isgn
# target : 1.85555

# --------------------------------------------------
ic.trm_isgn = 35
ic.trm_isofs = 0
# --------------------------------------------------

ic.write_byte(0x0c, 0x40)
ic.otp_lorng = 0

print(f"trm_isgn : {ic.trm_isgn}")
print(f"trm_isofs : {ic.trm_isofs}")
print(f"otp_lorng (0b) : {ic.otp_lorng}")
print(f"0x0c (0x40) : {ic.read_byte(0x0c):#04x}")
print(f"otp_lorng (0b) : {ic.otp_lorng}")

In [ ]:
ld.disable
ld.vset = 20
delay(1)
ld.enable

In [ ]:
# calculate R_cable
ld.vset = 19.5
delay(1)
iload_19p5 = dm2.current_10A
r_cable = 0.5 / iload_19p5
print(f"R_cable : {r_cable:.06f}")

ld.vset = 20
delay(1)

In [25]:
log.output_csv([])
log.output_csv([])
log.output_csv([])
log.output_csv(["trm_isgn", "trm_isofs", "transition current (A)"])

# target start point : 3A
start_vin = 20 - round(3 * r_cable, 3)

list_item = list(np.arange(18, start_vin, 0.001))
for_item  = [round(num, 3) for num in list_item]
for_item.reverse()

pre_iset = 0

for trim_bit in range(0b1_000):
    
    ic.trm_isofs = trim_bit

    for item_v in for_item:

        ld.vset = item_v
        
        read_i = abs(round(dm2.current_10A, 6))
        read_cocp = abs(round(dm1.voltage_10, 1))
        
        isgn = ic.trm_isgn
        print(f"[{isgn:#04x}, {trim_bit:#04x}] {read_i}, {read_cocp}")
        
        if read_cocp > 0.5:
            ld.vset = 20
            break
        else:
            pre_iset = round(read_i, 6)

    ld.vset = 20
    
    isgn = ic.trm_isgn
    print(f"[{isgn:#04x}, {trim_bit:#04x}] fine current : {pre_iset}")
    delay(1)
    
    log.output_csv([isgn, trim_bit, pre_iset])

# target : 3.25 + 0.022 = 3.272

[0x23, 0x00] 0.059032, 0.4
[0x23, 0x00] 0.059032, 0.0
[0x23, 0x00] 0.066264, 0.0
[0x23, 0x00] 2.570654, 0.0
[0x23, 0x00] 2.948443, 0.0
[0x23, 0x00] 2.962678, 0.0
[0x23, 0x00] 2.967647, 0.0
[0x23, 0x00] 2.970701, 0.0
[0x23, 0x00] 2.97596, 0.0
[0x23, 0x00] 2.978966, 0.0
[0x23, 0x00] 2.985022, 0.0
[0x23, 0x00] 2.989081, 0.0
[0x23, 0x00] 2.989081, 0.0
[0x23, 0x00] 2.99341, 0.0
[0x23, 0x00] 2.996901, 0.0
[0x23, 0x00] 3.000277, 0.0
[0x23, 0x00] 3.003781, 0.0
[0x23, 0x00] 3.011335, 0.0
[0x23, 0x00] 3.014502, 0.0
[0x23, 0x00] 3.018144, 0.0
[0x23, 0x00] 3.021098, 0.0
[0x23, 0x00] 3.025973, 0.0
[0x23, 0x00] 3.025973, 0.0
[0x23, 0x00] 3.032868, 0.0
[0x23, 0x00] 3.036405, 0.0
[0x23, 0x00] 3.03954, 0.0
[0x23, 0x00] 3.043504, 0.0
[0x23, 0x00] 3.046169, 0.0
[0x23, 0x00] 3.049792, 0.0
[0x23, 0x00] 3.05312, 0.0
[0x23, 0x00] 3.057573, 0.0
[0x23, 0x00] 3.060608, 0.0
[0x23, 0x00] 3.064501, 0.0
[0x23, 0x00] 3.066967, 0.0
[0x23, 0x00] 3.071168, 0.0
[0x23, 0x00] 3.074364, 0.0
[0x23, 0x00] 3.078712, 0.0
[0x23

In [ ]:
disable_all()
bench_isgn = 35
bench_isofs = 7

In [ ]:
# --------------------------------------------------
# ILIM_RANGE verification
# --------------------------------------------------

h = ic.get_i2c_handler()

# --------------------------------------------------
evb_no = "evb#1"
log.output_set_filename(f"[EL6200_A1] trm_isgn_{evb_no}")

dm2.current_10A
ld.iset = 0.01
# --------------------------------------------------

log.output_csv([])
log.output_csv([])
log.output_csv([])
log.output_csv(["ILIM_RANGE verification"])

sm.cfg_all = 3.3, 0.1
sm.power_recycle
ps.cfg_all = 12, 6.5
ps.power_recycle

print(f"trm_isgn : {ic.trm_isgn:#x} ({ic.trm_isgn})")
print(f"trm_isofs : {ic.trm_isofs:#x}")

# activate tst mode
ic.write_byte(0xa0, 0xf9)
ic.write_byte(0xa0, 0x9f)
print(f"Read byte : [0xa0]={ic.read_byte(0xa0):#04x}")

# --------------------------------------------------
ic.trm_isgn = bench_isgn
ic.trm_isofs = bench_isofs
print(f"trm_isgn : {ic.trm_isgn:#x} ({ic.trm_isgn})")
print(f"trm_isofs : {ic.trm_isofs:#x}")
# --------------------------------------------------

ic.ILIM_HYS = 0 # 0mA
ic.ILIM_TH = 0
ic.ILIM_TH_EX = 0
ic.ACTIVE_ILIM_EN = 0 # latch
ic.OC_FAULT_PIN_EN = 1

In [ ]:
# [voltage, pre_iout, ilim_th_ex, active_ilim_en, ilim_acc]
log.output_csv(["trm_isgn", "trm_isofs", "VIN (V)", "I_LOAD (A)", "ILIM_TH_EX", "ACTIVE_ILIM_EN", "Accuracy (%)"])

In [ ]:
vin = [20, 28]
# vin = [12]

# current : [ILIM_TH, ILIM_TH_EX]
dict_load = {
    3.3 : [2, 0],
    # 4.1 : [2, 1],
    5   : [27, 0],
    # 5.5 : [15, 1],
    5.9 : [31, 1]
}

dm2.current_10A

print(f"trm_isgn : {ic.trm_isgn:#x} ({ic.trm_isgn})")
print(f"trm_isofs : {ic.trm_isofs:#x}")

for voltage in vin:
    
    ps.cfg_all = voltage, 6.5
    delay(1)
    count = 0
    
    # activate tst mode
    ic.write_byte(0xa0, 0xf9)
    ic.write_byte(0xa0, 0x9f)
    print(f"Read byte : [0xa0]={ic.read_byte(0xa0):#04x}")
    
    ic.trm_isgn = bench_isgn
    ic.trm_isofs = bench_isofs
    print(f"trm_isgn : {ic.trm_isgn:#x} ({ic.trm_isgn})")
    print(f"trm_isofs : {ic.trm_isofs:#x}")
    
    for current, reg in dict_load.items():
        
        i_start  = current * 0.9
        i_finish = current + 0.2
        
        step = 0.003
        ndigit = 3

        list_temp = list(np.arange(i_start, i_finish, step))
        list_load = [round(num, ndigit) for num in list_temp]

        ic.ILIM_TH = reg[0]
        ic.ILIM_TH_EX = reg[1]
        
        ld.iset = list_load[0] * 0.2
        ld.enable
        delay(1)
        
        # soft-start
        for pre_load in [0.60, 0.65, 0.70, 0.75, 0.80, 0.85]:
            ld.iset = list_load[0] * pre_load
            delay(0.2)
        
        pre_iout = 0
        
        for set_load in list_load:
            
            ld.iset = set_load
            
            try:
                off_by_lim = ic.OFF_BY_ILIM
                sw_sts = ic.SW_STS
                ilim_hys = ic.ILIM_HYS
                ilim_th = ic.ILIM_TH
                ilim_th_ex = ic.ILIM_TH_EX
                active_ilim_en = ic.ACTIVE_ILIM_EN
            except:
                off_by_lim = "NACK"
                sw_sts = "NACK"
                ilim_hys = "NACK"
                ilim_th = "NACK"
                ilim_th_ex = "NACK"
                active_ilim_en = "NACK"
            
            iout = abs(round(dm2.current_10A, 6))
            
            # ret = [voltage, iout, sw_sts, off_by_lim, ilim_hys, ilim_th, ilim_th_ex, active_ilim_en]
            # log.output_csv(ret)
            
            print(f"[{current:.03f}] {iout:.06f} ({sw_sts:#x})")
            
            if sw_sts != 2:
                count += 1
                if count > 10:
                    ld.disable
                    delay(1)
                    break
            else:
                if pre_iout < iout:
                    pre_iout = iout
                else:
                    pass
        
        ilim_acc = round(((pre_iout-current)/current*100), 6)
        ret = [ic.trm_isgn, ic.trm_isofs, voltage, pre_iout, ilim_th_ex, active_ilim_en, ilim_acc]
        log.output_csv(ret)
        
        # h.gpio_7(0)
        # ld.disable
        # delay(1)
        # h.gpio_7(1)
        # delay(1)
        print(f"trm_isgn : {ic.trm_isgn:#x} ({ic.trm_isgn})")
        print(f"trm_isofs : {ic.trm_isofs:#x}")
        input("toggle EN")
        print("toggle EN")

# ----------------------------------------------------------------------------

print(f"trm_isgn : {ic.trm_isgn:#x} ({ic.trm_isgn})")
print(f"trm_isofs : {ic.trm_isofs:#x}")
# disable_all()

In [ ]:
disable_all()